#### **CREACIÓN DE TIPOS DE CAJAS**

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from importlib import reload

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")

Durante esta instancia, crearemos nuevos tipos de cajas con determinadas dimensiones internas a definir.

Notemos que por restricción, únicamente podemos variar en un 10% cada dimensión interna con respecto de la original. Busquemos entonces las dimensiones mínimas y máximas que hay actualmente, para luego crear todas las combinaciones posibles entre las medidas intermedias.

In [2]:
menor_dim_alto = especificaciones_cajas.groupby('caja_tipo_id')['caja_interior_alto'].mean().nsmallest(1)
menor_dim_alto_val = menor_dim_alto.values[0]
mayor_dim_alto = especificaciones_cajas.groupby('caja_tipo_id')['caja_interior_alto'].mean().nlargest(1)
mayor_dim_alto_val = mayor_dim_alto.values[0]

menor_dim_ancho = especificaciones_cajas.groupby('caja_tipo_id')['caja_interior_ancho'].mean().nsmallest(1)
menor_dim_ancho_val = menor_dim_ancho.values[0]
mayor_dim_ancho = especificaciones_cajas.groupby('caja_tipo_id')['caja_interior_ancho'].mean().nlargest(1)
mayor_dim_ancho_val = mayor_dim_ancho.values[0]

menor_dim_largo = especificaciones_cajas.groupby('caja_tipo_id')['caja_interior_largo'].mean().nsmallest(1)
menor_dim_largo_val = menor_dim_largo.values[0]
mayor_dim_largo = especificaciones_cajas.groupby('caja_tipo_id')['caja_interior_largo'].mean().nlargest(1)
mayor_dim_largo_val = mayor_dim_largo.values[0]

Además, nos serviría el valor de volumen interno mínimo, para descartar aquellas combinaciones donde no valen la factibilidad de volumen para el peor de los casos dentro de los productos.

In [3]:
menor_volumen_producto = catalogo_productos.groupby('codigo_producto')['dim_producto_volumen'].mean().nsmallest(1)
menor_volumen_producto_val = menor_volumen_producto.values[0]

Ahora sí, creamos los nuevos tipos de cajas.

In [4]:
cajas_nuevas = []
contador = 1
paso = 1 # Define los saltos de medida

valores_alto = np.arange(round(menor_dim_alto_val * 0.9), round(mayor_dim_alto_val * 1.1) + paso, paso)
valores_ancho = np.arange(round(menor_dim_ancho_val * 0.9), round(mayor_dim_ancho_val * 1.1) + paso, paso)
valores_largo = np.arange(round(menor_dim_largo_val * 0.9), round(mayor_dim_largo_val * 1.1) + paso, paso)

for alto in valores_alto:
    for ancho in valores_ancho:
        for largo in valores_largo:
            # Generar ID
            caja_id = f"CAJ{contador:07d}"  # CAJ0000001, CAJ0000002, ...
            
            # Crear caja
            caja = Caja(
                caja_id=caja_id,
                dim_interior_ancho=float(ancho),
                dim_interior_largo=float(largo),
                dim_interior_alto=float(alto)
            )
            
            # Chequeamos factibilidad mínimal
            if caja.volumen_interno() >= menor_volumen_producto_val:
                cajas_nuevas.append(caja)
                contador += 1

print(f"Se crearon {len(cajas_nuevas)} tipos de cajas.")

Se crearon 4327527 tipos de cajas.


In [5]:
cajas_nuevas[:5]

[<Caja CAJ0000001 | Int: 193.0 x 434.0 x 124.0mm | Compra Total: 0>,
 <Caja CAJ0000002 | Int: 193.0 x 435.0 x 124.0mm | Compra Total: 0>,
 <Caja CAJ0000003 | Int: 194.0 x 432.0 x 124.0mm | Compra Total: 0>,
 <Caja CAJ0000004 | Int: 194.0 x 433.0 x 124.0mm | Compra Total: 0>,
 <Caja CAJ0000005 | Int: 194.0 x 434.0 x 124.0mm | Compra Total: 0>]

Exportamos en csv los resultados:

In [6]:
df_cajas = pd.DataFrame([
    {
        'caja_tipo_id': caja.caja_id,
        'caja_interior_alto': caja.dim_interior_alto,
        'caja_interior_ancho': caja.dim_interior_ancho,
        'caja_interior_largo': caja.dim_interior_largo
    }
    for caja in cajas_nuevas
])

df_cajas.to_csv('4r.cajas_nuevas2.csv', index=False)